# Seguimiento completo del pipeline

Este notebook documenta las tres fases del modelo de machine learning:

1. **Entrenamiento** — el modelo aprende patrones a partir de datos históricos etiquetados (RECIBIDO = 0, ABANDONO/BAJA = 1)
2. **Validación** — se evalúa el rendimiento del modelo sobre datos que no vio durante el entrenamiento
3. **Pronóstico** — el modelo ya entrenado predice el riesgo de abandono sobre estudiantes activos cuyo desenlace aún se desconoce (CURSO/PAUSA)

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(os.path.join('..')))

from src.ingestor_datos import IngestorDatos
from src.preparador_datos import PreparadorDatos
from src.entrenador_modelos import EntrenadorModelos
from src.evaluador_riesgo import EvaluadorRiesgo

---
## Fase 1 — Entrenamiento (datos etiquetados)

Los **datos etiquetados** son aquellos cuyo estado académico final ya se conoce:
- `target_ml = 0` → estudiante **RECIBIDO** (continúa)
- `target_ml = 1` → estudiante **ABANDONO** o **BAJA** (no continúa)

Con estos datos el modelo aprende a reconocer qué patrones de notas, asistencia y datos sociodemográficos se asocian al abandono.

In [ ]:
# Carga de datos
ingestor = IngestorDatos()
datasets = ingestor.leer_datos()

# Preparación
preparador = PreparadorDatos(datasets)
df_master = preparador.ejecutar_preparacion()

# Dataset para ML
df_ml = preparador.preparar_dataset_ml(df_master)

# Filtrar solo datos etiquetados (target conocido)
df_etiquetados = df_ml.dropna(subset=['target_ml'])
total_etiquetados = len(df_etiquetados)
abandonos = int(df_etiquetados['target_ml'].sum())
continuan = total_etiquetados - abandonos

print('=== FASE 1: ENTRENAMIENTO (datos etiquetados) ===')
print(f'Total registros etiquetados: {total_etiquetados}')
print(f'  - Continúa (target=0): {continuan}')
print(f'  - Abandono (target=1): {abandonos}')
print(f'  - Proporción abandono: {abandonos/total_etiquetados:.1%}')
print()
print('Columnas predictoras disponibles para entrenar:')
print(list(df_etiquetados.drop(columns=['target_ml']).columns))

---
## Fase 2 — Validación (métricas por bimestre)

Para cada bimestre (del 1 al 6) se entrena un árbol de decisión independiente, usando solo las columnas de notas y asistencia disponibles hasta ese hito temporal. Se divide el conjunto etiquetado en:
- **80% entrenamiento** — el modelo aprende
- **20% prueba** — el modelo se evalúa con datos que no ha visto

Las métricas principales son:
- **Accuracy**: proporción de aciertos global
- **Recall (Abandono)**: proporción de abandonos reales que el modelo detecta correctamente. Es la métrica de negocio más importante porque buscamos minimizar los falsos negativos.

In [ ]:
# Entrenamiento con validación
entrenador = EntrenadorModelos()
df_metricas = entrenador.entrenar_y_guardar(df_ml, preparador)

print('=== FASE 2: VALIDACIÓN ===')
print()
print(df_metricas.to_string(index=False))
print()
print(f'Accuracy promedio: {df_metricas["Accuracy"].mean():.2f}')
print(f'Recall (Abandono) promedio: {df_metricas["Recall (Abandono)"][df_metricas["Recall (Abandono)"] > 0].mean():.2f}')

---
## Fase 3 — Pronóstico (datos sin etiqueta)

Los **datos sin etiqueta** corresponden a los estudiantes actualmente activos en estado **CURSO** o **PAUSA**, cuyo desenlace final (graduación o abandono) aún se desconoce.

El modelo entrenado (árbol de decisión) infiere su probabilidad de abandono basándose en los patrones aprendidos en la Fase 1. Los umbrales de decisión son:
- `probabilidad >= 0.70` → **ALTO**
- `probabilidad >= 0.40` → **MEDIO**
- `probabilidad < 0.40` → **BAJO**

In [ ]:
# Evaluación con el modelo guardado en disco
evaluador = EvaluadorRiesgo()
df_riesgo = evaluador.ejecutar_evaluacion(df_master)

print('=== FASE 3: PRONÓSTICO (datos sin etiqueta) ===')
print()

# Separar por tipo de predicción
df_pronostico = df_riesgo[df_riesgo['tipo_prediccion'] == 'PRONÓSTICO']
df_historico = df_riesgo[df_riesgo['tipo_prediccion'] == 'HISTÓRICO']

print(f'Alumnos históricos (target conocido): {len(df_historico)}')
print(f'Alumnos pronosticados (sin etiqueta): {len(df_pronostico)}')
print()

if len(df_pronostico) > 0:
    print('Distribución del riesgo pronosticado:')
    for nivel in ['ALTO', 'MEDIO', 'BAJO']:
        conteo = len(df_pronostico[df_pronostico['nivel_riesgo'] == nivel])
        print(f'  {nivel}: {conteo}')
    print()

# Mostrar alumnos con riesgo alto
alumnos_alto = df_pronostico[df_pronostico['nivel_riesgo'] == 'ALTO']
if len(alumnos_alto) > 0:
    print('Alumnos en riesgo ALTO (justificación XAI):')
    cols_show = ['n_siu', 'estudio', 'probabilidad_abandono', 'justificacion_riesgo']
    print(alumnos_alto[cols_show].to_string(index=False))

---
## Resumen del pipeline completo

Se han ejecutado las tres fases del modelo de machine learning:

| Fase | Datos | Target | Salida |
|---|---|---|---|
| 1. Entrenamiento | Históricos (RECIBIDO/BAJA) | Conocido (0/1) | Modelos entrenados por bimestre |
| 2. Validación | 20% de los históricos | Conocido (0/1) | Métricas (Accuracy, Recall) |
| 3. Pronóstico | Activos (CURSO/PAUSA) | Desconocido | Riesgo inferido + Justificación XAI |

Este diseño garantiza que los datos de pronóstico **nunca se mezclan** con los datos de entrenamiento, preservando la pureza experimental.